In [1]:
import importlib.util
import time
from pathlib import Path

import qi

NAO_IP = "192.168.1.102"
NAO_PORT = 9559

session = qi.Session()
session.connect(f"tcp://{NAO_IP}:{NAO_PORT}")
motion = session.service("ALMotion")
posture = session.service("ALRobotPosture")
behaviors = session.service("ALBehaviorManager")
tts = session.service("ALTextToSpeech")


def play_movement(name):
    path = Path("movements") / f"{name}.py"
    spec = importlib.util.spec_from_file_location(f"movement_{name}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    start_posture, _, raw = module.main()
    names = [joint[0] for joint in raw]
    times = [[float(keyframe[0]) for keyframe in joint[1]] for joint in raw]
    angles = [
        [[float(keyframe[1]), keyframe[2], keyframe[3]] for keyframe in joint[1]]
        for joint in raw
    ]
    motion.wakeUp()
    if start_posture:
        posture.goToPosture(start_posture, 0.5)
    motion.angleInterpolationBezier(names, times, angles)


def run_behavior(name):
    motion.wakeUp()
    behaviors.runBehavior(name)
    idle_checks = 0
    while idle_checks < 2:
        idle_checks = 0 if behaviors.isBehaviorRunning(name) else idle_checks + 1
        if idle_checks < 2:
            time.sleep(0.2)


def stop_all():
    for service, method in (
        (motion, "killAll"),
        (tts, "stopAll"),
        (behaviors, "stopAllBehaviors"),
    ):
        try:
            getattr(service, method)()
        except Exception as exc:
            print(f"{method}: {exc}")


print(f"已连接 NAO {NAO_IP}:{NAO_PORT}")


已连接 NAO 192.168.1.102:9559


In [2]:
motion.wakeUp()
posture.goToPosture("Crouch", 0.5)


True

In [3]:
motion.wakeUp()
posture.goToPosture("Stand", 0.5)


True

In [4]:
play_movement("taiji")


In [5]:
play_movement("littleapple")


In [6]:
try:
    while True:
        run_behavior("animations/Stand/Gestures/BowShort_1")
        run_behavior("animations/Stand/Gestures/Hey_1")
except KeyboardInterrupt:
    stop_all()


In [ ]:
stop_all()
